# Phase-Resolved Analysis: Ephemeris & Exposure Integration
This notebook demonstrates how to use the `PhaseEphemeris` protocol to assign phases to event data, select multiple specific phase intervals (like off-pulse regions), and strictly correct the mission exposure (livetime) for spectral fitting.

In [1]:
import numpy as np
from astropy.time import Time
from cosipy.util import fetch_wasabi_file

from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.phase_resolved_analysis.ephemeris import PulsarTimingModel
from cosipy.phase_resolved_analysis.phase_assigner import PhaseAssigner
from cosipy.phase_resolved_analysis.phase_selector import PhaseSelector

Welcome to JupyROOT 6.28/12


11:07:44 WARNING   The naima package is not available. Models that depend on it will not be         ]8;id=279420;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=276159;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  available                                                                                        

         WARNING   The GSL library or the pygsl wrapper cannot be loaded. Models that depend on it  ]8;id=955021;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=125569;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#65\65]8;;\
                  will not be available.                                                                           

         WARNING   The ebltable package is not available. Models that depend on it will not be     ]8;id=284745;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py\absorption.py]8;;\:]8;id=423656;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py#33\33]8;;\
                  available                                                                                        

11:07:44 INFO      Starting 3ML!                                                                     ]8;id=847398;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=696003;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py#44\44]8;;\

...

         WARNING   Could not import plugin FermiLATLike.py. Do you have the relative instrument     ]8;id=370656;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=277430;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py#126\126]8;;\
                  software installed and configured?                                                               

         WARNING   No fermitools installed                                              ]8;id=101763;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py\lat_transient_builder.py]8;;\:]8;id=542532;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py#44\44]8;;\

         WARNING   Env. variable OMP_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=512936;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=915222;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable MKL_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=171295;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=840551;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable NUMEXPR_NUM_THREADS is not set. Please set it to 1 for optimal     ]8;id=724604;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=542082;file:///home/abhi/anaconda3/envs/cosipy/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

In [2]:
# Download the real DC2 orientation file
print("Downloading the spacecraft orientation file (this may take a moment)...")
ori_file = fetch_wasabi_file(
    'COSI-SMEX/DC2/Data/Orientation/20280301_3_month_with_orbital_info.ori', 
    output='20280301_3_month_with_orbital_info.ori', 
    checksum='416fcc296fc37a056a069378a2d30cb2'
)

A file named 20280301_3_month_with_orbital_info.ori already exists with the specified checksum (416fcc296fc37a056a069378a2d30cb2). Skipping.


In [3]:
# Download the DC2 Crab FITS data (Unbinned)
# The checksum ensures we don't re-download if the file is already cached locally.
data_file = fetch_wasabi_file(
    'COSI-SMEX/DC2/Data/Sources/Crab_DC2_3months_unbinned_data.fits.gz', 
    output='Crab_DC2_3months_unbinned_data.fits.gz', 
    unzip=True, 
    checksum="539e432bc9843d20396dd6a210772b6e"
)
print(f"Data ready at: {data_file}")

A file named Crab_DC2_3months_unbinned_data.fits already exists with the specified checksum (539e432bc9843d20396dd6a210772b6e). Skipping.


Data ready at: None


In [4]:
# --- INITIALIZE TIMING MODEL ---
# Define the reference epoch (T0)
t0 = Time(59000.0, format='mjd', scale='tdb')

# Initialize the simple constant-frequency model
timing_model = PulsarTimingModel.from_par_file('crab.par', t0)
print(f"Loaded Pulsar Spin Frequency: {timing_model.f0:.6f}")

Loaded Pulsar Spin Frequency: 29.946923 Hz


In [5]:
# --- ASSIGN PHASES TO DATA ---
input_fits_file = "Crab_DC2_3months_unbinned_data.fits"
output_fits_file = "Crab_DC2_3months_unbinned_data_with_pulse_phase.fits"

# Pass the protocol object directly!
assigner = PhaseAssigner(timing_model)
assigner.add_phase_column(input_fits_file, output_fits_file)

print(f"Done! Created: {output_fits_file}")

Done! Created: Crab_DC2_3months_unbinned_data_with_pulse_phase.fits


In [9]:
# --- CORRECT MISSION EXPOSURE ---

# 1. Load the original spacecraft pointing history
ori_file = '20280301_3_month_with_orbital_info.ori'
print(f"Loading SpacecraftHistory from {ori_file}...")
sc_history = SpacecraftHistory.open(ori_file)

# Let's look at the first few bins of the original livetime
print("\n--- BEFORE CORRECTION ---")
print(f"Original Livetime Array (first 5 bins): \n{sc_history._livetime_hist.contents[:5]}")

# Define the phase cut
intervals = [
    (0.00, 0.35),  
    (0.70, 1.00),  
]
expected_fraction = sum([stop - start for start, stop in intervals])
print(f"\nPhase Cut Total Width: {expected_fraction * 100:.0f}%")

# 2. Apply the correction using the protocol
print("Applying phase exposure correction...")
sc_history.update_ephemeris(timing_model, intervals)

print("\n--- AFTER CORRECTION ---")
print(f"Corrected Livetime Array (first 5 bins): \n{sc_history._livetime_hist.contents[:5]}")
print(f"\nExposure is now perfectly scaled to {expected_fraction * 100:.0f}%!")

Loading SpacecraftHistory from 20280301_3_month_with_orbital_info.ori...

--- BEFORE CORRECTION ---
Original Livetime Array (first 5 bins): 
[1. 1. 1. 1. 1.] s

Phase Cut Total Width: 65%
Applying phase exposure correction...

--- AFTER CORRECTION ---
Corrected Livetime Array (first 5 bins): 
[0.65 0.65 0.65 0.65 0.65] s

Exposure is now perfectly scaled to 65%!
